# 以当前的数据给未来选股

1、获取当前行情数据
2、财务数据需要提前存储，否则会出现联网问题，年报久久获取一次就行，需要体检存储为表格数据
3、

In [7]:
import akshare as ak
import pandas as pd
import time
import traceback
from tqdm import tqdm

nianbao_df = pd.read_csv("../data/A股_日行情_年报_流通市值.csv", dtype={"股票代码": str})
def load_latest_financial_single(stock_code, trade_date):
    trade_date = pd.to_datetime(trade_date)

    sub = nianbao_df[
        (nianbao_df["股票代码_full"] == stock_code) &
        (pd.to_datetime(nianbao_df["日期"]) <= trade_date)
    ]

    if sub.empty:
        return None

    sub = sub.sort_values("NOTICE_DATE")
    return sub.iloc[-1][["NOTICE_DATE", "EPSJB", "ROEJQ"]]


def load_daily_price(symbol: str, trade_date: pd.Timestamp):
    df = ak.stock_zh_a_hist_min_em(
        symbol=symbol,
        start_date=trade_date.strftime("%Y%m%d"),
        period="5",
        adjust=""
    )

    if df.empty:
        return None

    return df.iloc[-1]

def load_market_value(symbol: str, trade_date: pd.Timestamp):
    df = ak.stock_value_em(symbol=symbol)
    df["数据日期"] = pd.to_datetime(df["数据日期"])
    df = df[df["数据日期"] <= trade_date]

    if df.empty:
        return None

    return df.sort_values("数据日期").iloc[-1]["流通市值"]

def fetch_with_retry(fetch_func, max_retry=5, sleep_sec=1, desc=""):
    """
    通用重试封装
    :param fetch_func: 不带参数的函数（lambda）
    :param max_retry: 最大重试次数
    :param sleep_sec: 每次失败后的等待时间
    :param desc: 日志描述
    """
    for i in range(1, max_retry + 1):
        try:
            result = fetch_func()
            if result is not None:
                return result
            else:
                raise ValueError("返回 None")
        except Exception as e:
            print(f"[重试 {i}/{max_retry}] {desc} 失败：{e}")
            if i < max_retry:
                time.sleep(sleep_sec)
    return None

def select_stocks_on_date(
    trade_date: str,
    stock_list_csv="../data/stock_zh_a_spot_em_filtered_with_suffix.csv",
    select_stock_num=5
):
    
    trade_date = pd.to_datetime(trade_date)

    stock_df = pd.read_csv(stock_list_csv, dtype={"代码": str})

    results = []

    # 需要进行流通市值的股票
    stock_list = []
    
    for _, row in tqdm(
            stock_df.iterrows(),
            total=len(stock_df),
            desc="处理股票",
            ncols=100
        ):
        symbol = row["代码"]
        symbol_full = row["代码_full"]

        # print(f"Processing {symbol_full}...")

        # 筛选掉代码以 300、301、688、689、8、92、43 开头的股票
        if symbol.startswith(("300", "301", "688", "689", "8", "92", "43")):
            # print(f"筛选掉代码以 300、301、688、689、8、92、43 开头的股票  Skipped by code filter.")
            continue

        # ========= 财务 =========
        fin = load_latest_financial_single(symbol_full, trade_date)
        if fin is None:
            # print(f"未找到财务数据  Skipped by financial data missing.")
            continue

        eps = fin["EPSJB"].squeeze()
        roe = fin["ROEJQ"].squeeze()
        
        if pd.isna(eps) or pd.isna(roe):
            # print(f"财务数据缺失  Skipped by financial data incomplete.")
            continue

        # ========= 基本面过滤 =========
        if eps <= 0 or roe <= 10:
            # print(f"不符合基本面条件eps{eps}/roe{roe}  Skipped by fundamental filter.")
            continue

        # # ========= 行情 =========
        # price_row = fetch_with_retry(
        #     lambda: load_daily_price(symbol, trade_date),
        #     max_retry=5,
        #     sleep_sec=1,
        #     desc=f"{symbol} 行情数据"
        # )
        # if price_row is None:
        #     print(f"未找到行情数据  Skipped by price data missing.")
        #     continue

        # close_price = price_row["收盘"]

        # ========= 流通市值 =========
        # mv = fetch_with_retry(
        #     lambda: load_market_value(symbol, trade_date),
        #     max_retry=5,
        #     sleep_sec=1,
        #     desc=f"{symbol} 市值数据"
        # )
        # if mv is None:
        #     print(f"未找到市值数据  Skipped by market value data missing.")
        #     continue
        stock_list.append(symbol)

        results.append({
            "股票代码": symbol,
            "股票名称": row["名称"],
            # "收盘价": close_price,
            "EPS": eps,
            "ROE": roe,
            # "流通市值": mv
        })

    if not results:
        return pd.DataFrame()

    df = pd.DataFrame(results)

    # ========= 小市值排序 =========
    df = df.sort_values("流通市值")

    # 保存stock_list
    stock_list_df = pd.DataFrame({"股票代码": stock_list})
    stock_list_df.to_csv(f"../data/stock_list_{trade_date.strftime('%Y%m%d')}.csv", index=False)

    return df.head(select_stock_num)

selected_df = select_stocks_on_date(
    trade_date="2025-12-25",
    select_stock_num=5
)

print("2025-12-25 选股结果:")
print(selected_df)

# today = pd.Timestamp.today().normalize()
# # today加 09:30:00
# today = today + pd.Timedelta(hours=9, minutes=30)

# selected_df = select_stocks_on_date(
#     trade_date=today,
#     select_stock_num=5
# )
# print(f"{today.date()} 选股结果:")
# print(selected_df)


处理股票: 100%|█████████████████████████████████████████████████| 3403/3403 [42:34<00:00,  1.33it/s]


KeyError: '流通市值'

In [2]:
def load_daily_price(symbol: str, trade_date: pd.Timestamp):
    df = ak.stock_zh_a_hist_min_em(
        symbol=symbol,
        start_date=trade_date.strftime("%Y%m%d"),
        period="5",
        adjust=""
    )

    if df.empty:
        return None

    return df.iloc[-1]

df = load_daily_price("600714", pd.Timestamp("2025-12-25 09:30:00"))
df

时间     2026-01-08 15:00:00
开盘                   12.53
收盘                   12.55
最高                   12.55
最低                   12.52
涨跌幅                   0.16
涨跌额                   0.02
成交量                    879
成交额              1101760.0
振幅                    0.24
换手率                   0.03
Name: 431, dtype: object

In [32]:
def load_latest_financial_single(stock_code, trade_date):
    trade_date = pd.to_datetime(trade_date)

    sub = nianbao_df[
        (nianbao_df["股票代码_full"] == stock_code) &
        (pd.to_datetime(nianbao_df["日期"]) <= trade_date)
    ]

    if sub.empty:
        return None

    sub = sub.sort_values("NOTICE_DATE")
    return sub.iloc[-1][["NOTICE_DATE", "EPSJB", "ROEJQ"]]
df = load_latest_financial_single("000001.SZ", "2025-12-25")
df

NOTICE_DATE    2025-03-15
EPSJB                2.15
ROEJQ               10.08
Name: 5113226, dtype: object